# 第 8 讲：预测模型基础

> 落成预测模型示例代码：趋势外推、回归预测和 GM(1,1) 灰色预测。

## 使用说明

- 先从上到下运行全部单元。
- 每讲都会生成一个 `outputs/lesson-xx/` 目录，用于保存图表、表格或报告片段。
- 示例数据均为教学用合成数据，真实比赛中应替换为题目附件数据。

In [ ]:
LESSON_ID = "lesson-08"

from pathlib import Path
import os
import math
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

OUTPUT_ROOT = Path(os.environ.get("COURSE_OUTPUT_DIR", REPO_ROOT / "outputs")) / LESSON_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)
print(f"Output directory: {OUTPUT_ROOT}")

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error

t = np.arange(1, 37)
series = pd.Series(80 + 2.2 * t + 8 * np.sin(t / 3) + rng.normal(0, 3, len(t)), index=t, name="demand")
train = series.iloc[:28]
test = series.iloc[28:]

coef = np.polyfit(train.index, train.values, deg=1)
linear_pred = pd.Series(np.polyval(coef, test.index), index=test.index)
rolling_pred = pd.Series([train.tail(4).mean()] * len(test), index=test.index)

def gm11_forecast(x0, steps):
    x0 = np.array(x0, dtype=float)
    x1 = np.cumsum(x0)
    z1 = 0.5 * (x1[1:] + x1[:-1])
    B = np.column_stack([-z1, np.ones(len(z1))])
    Y = x0[1:]
    a, b = np.linalg.lstsq(B, Y, rcond=None)[0]
    def x1_hat(k):
        return (x0[0] - b / a) * np.exp(-a * k) + b / a
    x_hat = np.array([x1_hat(k) - x1_hat(k - 1) for k in range(1, len(x0) + steps)])
    return x_hat[-steps:]

gm_pred = pd.Series(gm11_forecast(train.values, len(test)), index=test.index)
metrics = pd.DataFrame({
    "model": ["linear_trend", "rolling_mean", "GM11"],
    "MAPE": [
        mean_absolute_percentage_error(test, linear_pred),
        mean_absolute_percentage_error(test, rolling_pred),
        mean_absolute_percentage_error(test, gm_pred),
    ],
})
metrics.to_csv(OUTPUT_ROOT / "forecast_metrics.csv", index=False, encoding="utf-8-sig")
metrics

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
series.plot(ax=ax, label="observed")
linear_pred.plot(ax=ax, label="linear trend")
rolling_pred.plot(ax=ax, label="rolling mean")
gm_pred.plot(ax=ax, label="GM(1,1)")
ax.axvline(test.index[0], color="gray", linestyle="--", label="test start")
ax.set_title("Forecasting model comparison")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "forecast_comparison.png", dpi=160)
plt.show()